<a href="https://colab.research.google.com/github/shankar791/CHITTI.AI/blob/main/197.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio python-multipart

In [ ]:
%%writefile /content/geochat_server.py

import io
import time
import torch

from PIL import Image
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from geochat.model.builder import load_pretrained_model
from geochat.mm_utils import process_images, tokenizer_image_token
from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from geochat.conversation import conv_templates

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

MODEL_PATH = "MBZUAI/geochat-7B"
MODEL_NAME = "geochat-7B"

DEVICE = "cuda"

print("Loading GeoChat...")
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this service.")

print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------------------------------------
# Load GeoChat ONCE at startup
# ---------------------------------------------------------

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_PATH,
    model_base=None,
    model_name=MODEL_NAME,
    load_8bit=True,
    load_4bit=False,
    device="cuda",
)

model.eval()

# Known-good GeoChat vision configuration
image_processor.size = {"shortest_edge": 504}
image_processor.crop_size = {
    "height": 504,
    "width": 504,
}

print("GeoChat loaded.")
print("Model:", model.__class__.__name__)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Image processor:", image_processor.__class__.__name__)
print("Context length:", context_len)
print("Parameters:", sum(p.numel() for p in model.parameters()))


# ---------------------------------------------------------
# Inference
# ---------------------------------------------------------

@torch.inference_mode()
def run_vqa(image: Image.Image, question: str):

    image = image.convert("RGB")

    # Conversation
    conv = conv_templates["llava_v1"].copy()

    prompt = DEFAULT_IMAGE_TOKEN + "\n" + question

    conv.append_message(conv.roles[0], prompt)
    conv.append_message(conv.roles[1], None)

    full_prompt = conv.get_prompt()

    # Text + image tokens
    input_ids = tokenizer_image_token(
        full_prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt",
    ).unsqueeze(0).to(DEVICE)

    # Image preprocessing
    image_tensor = process_images(
        [image],
        image_processor,
        model.config,
    )[0]

    image_tensor = image_tensor.unsqueeze(0).to(
        device=DEVICE,
        dtype=model.dtype,
    )

    print("\n--- REQUEST ---")
    print("Image:", image.size)
    print("Image tensor:", tuple(image_tensor.shape))
    print("Image dtype:", image_tensor.dtype)
    print("Input IDs:", tuple(input_ids.shape))

    start = time.perf_counter()

    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=[image.size],
        do_sample=False,
        max_new_tokens=256,
        use_cache=True,
    )

    latency_ms = round(
        (time.perf_counter() - start) * 1000,
        2,
    )

    generated_tokens = output_ids[0][input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    print("Generated tokens:", len(generated_tokens))
    print("Latency ms:", latency_ms)
    print("Answer:", answer)

    return {
        "task": "vqa",
        "model": "GeoChat-7B",
        "answer": answer,
        "latency_ms": latency_ms,
        "image_size": list(image.size),
        "processed_size": [504, 504],
    }


# ---------------------------------------------------------
# FastAPI
# ---------------------------------------------------------

app = FastAPI(title="SatQuery GeoChat API")


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": model is not None,
        "model": "GeoChat-7B",
        "model_class": model.__class__.__name__,
        "device": DEVICE,
    }


@app.post("/vqa")
async def vqa(
    image: UploadFile = File(...),
    question: str = Form(...),
):

    if not question.strip():
        raise HTTPException(
            status_code=400,
            detail="question is required",
        )

    try:
        data = await image.read()

        if not data:
            raise HTTPException(
                status_code=400,
                detail="empty image",
            )

        pil_image = Image.open(
            io.BytesIO(data)
        ).convert("RGB")

        return run_vqa(
            pil_image,
            question,
        )

    except HTTPException:
        raise

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=str(exc),
        )

Overwriting /content/geochat_server.py


In [ ]:
%%writefile /content/geochat_api.py

import sys

# Make the official GeoChat repository importable
sys.path.insert(0, "/content/GeoChat")

import torch
from fastapi import FastAPI

from geochat.model.builder import load_pretrained_model

app = FastAPI(title="SatQuery AI - GeoChat")


MODEL_PATH = "MBZUAI/geochat-7B"
MODEL_NAME = "geochat-7B"
DEVICE = "cuda"


print("======================================")
print("Starting GeoChat API")
print("======================================")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

print("GPU:", torch.cuda.get_device_name(0))


print("\nLoading GeoChat-7B...")

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_PATH,
    model_base=None,
    model_name=MODEL_NAME,
    load_8bit=True,
    load_4bit=False,
    device_map="auto",
    device=DEVICE,
)

model.eval()

# Critical GeoChat vision configuration
image_processor.size = {
    "shortest_edge": 504
}

image_processor.crop_size = {
    "height": 504,
    "width": 504
}


print("\n======================================")
print("GeoChat loaded successfully")
print("======================================")
print("Model:", model.__class__.__name__)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Image processor:", image_processor.__class__.__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Context length:", context_len)
print("Device map:", getattr(model, "hf_device_map", None))


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": model is not None,
        "model": "MBZUAI/geochat-7B",
        "model_class": model.__class__.__name__,
        "device": DEVICE
    }

Writing /content/geochat_api.py


In [ ]:
import subprocess
import time

# Start FastAPI/Uvicorn in background using the working Python 3.10 environment
cmd = [
    "/content/geochat-env/bin/python",
    "-m",
    "uvicorn",
    "geochat_api:app",
    "--host",
    "0.0.0.0",
    "--port",
    "8000",
    "--app-dir",
    "/content",
]

log_file = open("/content/geochat_server.log", "w")

process = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Started GeoChat API process:", process.pid)
print("Waiting for startup...")
time.sleep(5)

print("\n=== SERVER LOG ===")
with open("/content/geochat_server.log", "r") as f:
    print(f.read()[-5000:])

Started GeoChat API process: 36816
Waiting for startup...

=== SERVER LOG ===



In [ ]:
print("=== GEOCHAT SERVER LOG ===")

try:
    with open("/content/geochat_server.log", "r") as f:
        log = f.read()

    print(log if log else "[LOG IS EMPTY]")

except FileNotFoundError:
    print("Log file not found.")

=== GEOCHAT SERVER LOG ===
/content/geochat-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/content/geochat-env/lib/python3.10/site-packages/transformers/modeling_utils.py:541: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/content/geochat-env/lib/python3.10/site-packages/transformers/modeling_utils.py:541: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the curr

In [ ]:
!ps aux | grep geochat_api | grep -v grep || true
!fuser -v 8000/tcp 2>/dev/null || true

root       36816 21.3 10.4 18590236 1389204 ?    Sl   10:52   0:31 /content/geochat-env/bin/python -m uvicorn geochat_api:app --host 0.0.0.0 --port 8000 --app-dir /content
 36816

In [ ]:
import requests
import time

for attempt in range(5):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=5
        )

        print("STATUS:", response.status_code)
        print("RESPONSE:", response.json())
        break

    except Exception as e:
        print(f"Attempt {attempt + 1}/5:", type(e).__name__, e)
        time.sleep(3)

STATUS: 200
RESPONSE: {'status': 'ok', 'model_loaded': True, 'model': 'MBZUAI/geochat-7B', 'model_class': 'GeoChatLlamaForCausalLM', 'device': 'cuda'}


In [ ]:
# Append VQA endpoint to the existing server file.

with open("/content/geochat_api.py", "a") as f:
    f.write(r'''

import io
import time
from PIL import Image
from fastapi import File, Form, HTTPException, UploadFile

from geochat.mm_utils import process_images, tokenizer_image_token
from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from geochat.conversation import conv_templates


@torch.inference_mode()
def run_vqa(image: Image.Image, question: str):

    image = image.convert("RGB")

    conv = conv_templates["llava_v1"].copy()

    prompt = DEFAULT_IMAGE_TOKEN + "\n" + question

    conv.append_message(conv.roles[0], prompt)
    conv.append_message(conv.roles[1], None)

    full_prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(
        full_prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt",
    ).unsqueeze(0).to(DEVICE)

    image_tensor = process_images(
        [image],
        image_processor,
        model.config,
    )[0]

    image_tensor = image_tensor.unsqueeze(0).to(
        device=DEVICE,
        dtype=model.dtype,
    )

    # Ensure the known-good GeoChat vision resolution.
    # The image processor is already configured to 504x504.
    start = time.perf_counter()

    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=[image.size],
        do_sample=False,
        max_new_tokens=256,
        use_cache=True,
    )

    latency_ms = round(
        (time.perf_counter() - start) * 1000,
        2,
    )

    # Decode only newly generated tokens.
    generated_tokens = output_ids[0][input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    return {
        "task": "vqa",
        "model": "GeoChat-7B",
        "answer": answer,
        "latency_ms": latency_ms,
        "image_size": list(image.size),
        "processed_size": [504, 504],
    }


@app.post("/vqa")
async def vqa(
    image: UploadFile = File(...),
    question: str = Form(...),
):

    if not question.strip():
        raise HTTPException(
            status_code=400,
            detail="Question cannot be empty.",
        )

    try:
        image_bytes = await image.read()

        if not image_bytes:
            raise HTTPException(
                status_code=400,
                detail="Empty image.",
            )

        pil_image = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")

        return run_vqa(
            pil_image,
            question,
        )

    except HTTPException:
        raise

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e),
        )
''')

print("VQA endpoint code added to file.")

VQA endpoint code added to file.


In [ ]:
import time

print("Waiting for GeoChat model loading...")

for i in range(12):
    time.sleep(10)

    try:
        with open("/content/geochat_server.log", "r") as f:
            log = f.read()

        print(f"\n--- Check {i+1}/12 ---")
        print(log[-1200:])

        if "Uvicorn running on" in log:
            print("\n✅ SERVER IS READY")
            break

        if "Traceback" in log or "ERROR:" in log:
            print("\n❌ SERVER ERROR DETECTED")
            break

    except Exception as e:
        print("Could not read log:", e)

Waiting for GeoChat model loading...

--- Check 1/12 ---
._load_from_state_dict(*args)
/content/geochat-env/lib/python3.10/site-packages/transformers/modeling_utils.py:541: UserWarning: for vision_model.post_layernorm.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
Starting GeoChat API
Python: 3.10.12 (main, Jul 15 2026, 23:40:17) [GCC 11.4.0]
PyTorch: 2.13.0+cu130
CUDA: True
GPU: Tesla T4

Loading GeoChat-7B...
Loading GeoChat......

Loading checkpoint shards: 100%|██████████| 2/2 [01:02<00:00, 31.48s/it]
INFO:     Started server process [38039]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ SERVER IS READY


In [180]:
import subprocess

log = open("/content/geochat_server.log", "w")

process = subprocess.Popen(
    [
        "/content/geochat-env/bin/python",
        "-m",
        "uvicorn",
        "geochat_api:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
        "--app-dir",
        "/content",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("API restarted, PID:", process.pid)

API restarted, PID: 52267


In [181]:
print("=== CURRENT API LOG ===")

try:
    with open("/content/geochat_server.log", "r") as f:
        log = f.read()

    print(log[-10000:])

except Exception as e:
    print("Could not read log:", e)

=== CURRENT API LOG ===



In [ ]:
import subprocess
import time
import re

tunnel = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None

for _ in range(60):
    line = tunnel.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line
        )

        if match:
            public_url = match.group(0)
            break

    time.sleep(0.5)

print("\nPUBLIC URL:")
print(public_url)

2026-08-30T11:07:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-30T11:07:41Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-30T11:07:45Z INF +--------------------------------------------------------------------------------------------+
2026-08-30T11:07:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-30T11:07:45Z INF |  https://promotes-march-mon-gray.trycloudflare.com    

In [179]:
from google.colab import output
url = output.eval_js("google.colab.kernel.proxyPort(8000)")
print("COLAB PUBLIC URL:")
print(url)

COLAB PUBLIC URL:
https://8000-gpu-t4-s-kkb-usw1b2-3tgmpenjbjomv-b.us-west1-2.prod.colab.dev


In [178]:
import requests

BASE = "http://127.0.0.1:8000"

health = requests.get(BASE + "/health", timeout=10)

print("Health:", health.status_code)
print(health.text)

print("\nEndpoints:")
openapi = requests.get(BASE + "/openapi.json", timeout=10).json()

for path, methods in openapi.get("paths", {}).items():
    print(path, list(methods.keys()))

Health: 200
{"status":"ok","model_loaded":true,"model":"MBZUAI/geochat-7B","model_class":"GeoChatLlamaForCausalLM","device":"cuda"}

Endpoints:
/health ['get']
/vqa ['post']
